# Introducción a NLP: Preprocesamiento de Texto con PyTorch

En esta notebook, nos enfocaremos en el preprocesamiento de texto, un paso fundamental en cualquier proyecto de Procesamiento de Lenguaje Natural (NLP). Antes de sumergirnos en arquitecturas complejas como RNNs o Transformers, es crucial dominar cómo preparar los datos de texto para que sean utilizados eficazmente por los modelos de aprendizaje profundo.

## Intro

### Objetivos

1. **Explorar técnicas clave de preprocesamiento de texto**, incluyendo tokenización, limpieza, y manejo de vocabulario.
2. **Comprender la importancia del padding y el truncamiento** en el manejo de secuencias de texto de longitud variable.
3. **Convertir texto en representaciones numéricas** adecuadas para ser ingresadas en modelos de NLP.
4. **Implementar un pipeline de preprocesamiento** en PyTorch que prepare el texto para futuras etapas de modelado.

### Contenido

1. Introducción al concepto de preprocesamiento en NLP y su relevancia.
2. Limpieza y tokenización del texto utilizando bibliotecas estándar.
3. Construcción de un vocabulario a partir de datos textuales.
4. Conversión de texto en índices numéricos para su uso en modelos.
5. Implementación de técnicas de padding y truncamiento.
6. Preparación del dataset para ser utilizado en modelos de aprendizaje profundo en PyTorch.

### Sobre el Dataset IMDB

En esta notebook utilizaremos el dataset de reseñas de películas de IMDB, un conjunto de datos ampliamente utilizado en la investigación de NLP. El dataset contiene 50,000 reseñas de películas en inglés, etiquetadas como **positivas** o **negativas**. Se divide equitativamente en un conjunto de entrenamiento y un conjunto de prueba, con 25,000 reseñas en cada uno. Las reseñas positivas y negativas están equilibradas, lo que lo convierte en un excelente recurso para entrenar y evaluar modelos de análisis de sentimiento.

El dataset fue creado por Andrew Maas y sus colegas en la Universidad de Stanford, y está disponible públicamente [aquí](https://ai.stanford.edu/~amaas/data/sentiment/). El propósito principal de este conjunto de datos es facilitar la investigación en tareas de clasificación de texto, como el análisis de sentimientos, donde el objetivo es predecir si una reseña es positiva o negativa basándose en su contenido textual.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

from torchinfo import summary

import pandas as pd
import numpy as np

import os
import sys
import tarfile
import urllib.request
import re
from pathlib import Path
from collections import Counter
from itertools import chain

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

from utils import (
    train,
)

In [2]:
print("Ruta(s) de descarga de datos NLTK:")
for p in nltk.data.path:
    print("  -", p)

# Descargar recursos necesarios
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)  # Open Multilingual Wordnet para lemmatization
nltk.download('averaged_perceptron_tagger', quiet=True)  # Para POS tagging

print("✅ Recursos de NLTK descargados correctamente")


Ruta(s) de descarga de datos NLTK:
  - /Users/rodrigobenitez/nltk_data
  - /Users/rodrigobenitez/Documents/GitHub/ORT-AI/.venv/nltk_data
  - /Users/rodrigobenitez/Documents/GitHub/ORT-AI/.venv/share/nltk_data
  - /Users/rodrigobenitez/Documents/GitHub/ORT-AI/.venv/lib/nltk_data
  - /usr/share/nltk_data
  - /usr/local/share/nltk_data
  - /usr/lib/nltk_data
  - /usr/local/lib/nltk_data
✅ Recursos de NLTK descargados correctamente


In [3]:
# Fijamos la semilla para que los resultados sean reproducibles
SEED = 23

torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [4]:
# definimos el dispositivo que vamos a usar
DEVICE = "cpu"  # por defecto, usamos la CPU
if torch.cuda.is_available():
    DEVICE = "cuda"  # si hay GPU, usamos la GPU
elif torch.backends.mps.is_available():
    DEVICE = "mps"  # si no hay GPU, pero hay MPS, usamos MPS
elif torch.xpu.is_available():
    DEVICE = "xpu"  # si no hay GPU, pero hay XPU, usamos XPU

print(f"Usando {DEVICE}")

NUM_WORKERS = 0 # Win y MacOS pueden tener problemas con múltiples workers
if sys.platform == 'linux':
    NUM_WORKERS = 4  # numero de workers para cargar los datos (depende de cada caso)

print(f"Usando {NUM_WORKERS}")

Usando mps
Usando 0


In [5]:
BATCH_SIZE = 512  # tamaño del batch

## Carga de Datos

Primero, descargaremos el [dataset IMDB](https://ai.stanford.edu/~amaas/data/sentiment/) y lo cargaremos en un DataFrame de Pandas para su fácil manipulación.

Se nos presentan dos carpetas: `train` y `test`, cada una con subcarpetas `pos` y `neg` que contienen reseñas positivas y negativas, respectivamente. Cada reseña se almacena en un archivo de texto separado. Utilizaremos la biblioteca `os` para navegar por las carpetas y cargar las reseñas en un DataFrame de Pandas.

In [6]:
DATA_PATH = "data"

url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
tar_path = os.path.join(DATA_PATH, "aclImdb_v1.tar.gz")

# Create data directory if it doesn't exist
os.makedirs(DATA_PATH, exist_ok=True)

# Download the file if not already downloaded
if not os.path.exists(tar_path):
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, tar_path)
    print("Download complete.")
else:
    print("File already downloaded.")

# Extract the tar.gz file
print("Extracting files...")
with tarfile.open(tar_path, "r:gz") as tar:
    tar.extractall(path=DATA_PATH)
print("Extraction complete.")

# Optional: print extracted folder contents
extracted_path = os.path.join(DATA_PATH, "aclImdb")
print(f"Dataset extracted to: {extracted_path}")
print("Contents:", os.listdir(extracted_path))

File already downloaded.
Extracting files...


/var/folders/_q/twq0th115qx2v66s9dcxxpp40000gn/T/ipykernel_48201/781029591.py:20: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=DATA_PATH)


Extraction complete.
Dataset extracted to: data/aclImdb
Contents: ['imdbEr.txt', 'test', 'imdb.vocab', 'README', 'train']


In [7]:
TRAIN_PATH = str(Path(extracted_path) / "train")
TEST_PATH = str(Path(extracted_path) / "test")

In [8]:
# Cargar el dataset de IMDB desde archivos locales
def load_imdb_data(base_directory):
    data = []
    for label in ["pos", "neg"]:
        folder = os.path.join(base_directory, label)
        for file in os.listdir(folder):
            with open(os.path.join(folder, file), "r", encoding="utf-8") as f:
                data.append((f.read(), 1 if label == "pos" else 0)) # 1 para positivo, 0 para negativo
    return pd.DataFrame(data, columns=["review", "sentiment"])

train_df = load_imdb_data(TRAIN_PATH)
test_df = load_imdb_data(TEST_PATH)

In [9]:
# Mostrar más caracteres por columna
pd.set_option('display.max_colwidth', None)
# Mostrar más columnas y filas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 0)  # 0 = autoajuste según la terminal

In [10]:
train_df.head()

,review,sentiment
0,"For a movie that gets no respect there sure are a lot of memorable quotes listed for this gem. Imagine a movie where Joe Piscopo is actually funny! Maureen Stapleton is a scene stealer. The Moroni character is an absolute scream. Watch for Alan ""The Skipper"" Hale jr. as a police Sgt.",1
1,"Bizarre horror movie filled with famous faces but stolen by Cristina Raines (later of TV's ""Flamingo Road"") as a pretty but somewhat unstable model with a gummy smile who is slated to pay for her attempted suicides by guarding the Gateway to Hell! The scenes with Raines modeling are very well captured, the mood music is perfect, Deborah Raffin is charming as Cristina's pal, but when Raines moves into a creepy Brooklyn Heights brownstone (inhabited by a blind priest on the top floor), things really start cooking. The neighbors, including a fantastically wicked Burgess Meredith and kinky couple Sylvia Miles & Beverly D'Angelo, are a diabolical lot, and Eli Wallach is great fun as a wily police detective. The movie is nearly a cross-pollination of ""Rosemary's Baby"" and ""The Exorcist""--but what a combination! Based on the best-seller by Jeffrey Konvitz, ""The Sentinel"" is entertainingly spooky, full of shocks brought off well by director Michael Winner, who mounts a thoughtfully downbeat ending with skill. ***1/2 from ****",1
2,"A solid, if unremarkable film. Matthau, as Einstein, was wonderful. My favorite part, and the only thing that would make me go out of my way to see this again, was the wonderful scene with the physicists playing badmitton, I loved the sweaters and the conversation while they waited for Robbins to retrieve the birdie.",1
3,"It's a strange feeling to sit alone in a theater occupied by parents and their rollicking kids. I felt like instead of a movie ticket, I should have been given a NAMBLA membership.<br /><br />Based upon Thomas Rockwell's respected Book, How To Eat Fried Worms starts like any children's story: moving to a new town. The new kid, fifth grader Billy Forrester was once popular, but has to start anew. Making friends is never easy, especially when the only prospect is Poindexter Adam. Or Erica, who at 4 1/2 feet, is a giant.<br /><br />Further complicating things is Joe the bully. His freckled face and sleeveless shirts are daunting. He antagonizes kids with the Death Ring: a Crackerjack ring that is rumored to kill you if you're punched with it. But not immediately. No, the death ring unleashes a poison that kills you in the eight grade.<br /><br />Joe and his axis of evil welcome Billy by smuggling a handful of slimy worms into his thermos. Once discovered, Billy plays it cool, swearing that he eats worms all the time. Then he throws them at Joe's face. Ewww! To win them over, Billy reluctantly bets that he can eat 10 worms. Fried, boiled, marinated in hot sauce, squashed and spread on a peanut butter sandwich. Each meal is dubbed an exotic name like the ""Radioactive Slime Delight,"" in which the kids finally live out their dream of microwaving a living organism.<br /><br />If you've ever met me, you'll know that I have an uncontrollably hearty laugh. I felt like a creep erupting at a toddler whining that his ""dilly dick"" hurts. But Fried Worms is wonderfully disgusting. Like a G-rated Farrelly brothers film, it is both vomitous and delightful.<br /><br />Writer/director Bob Dolman is also a savvy storyteller. To raise the stakes the worms must be consumed by 7 pm. In addition Billy holds a dark secret: he has an ultra-sensitive stomach.<br /><br />Dolman also has a keen sense of perspective. With such accuracy, he draws on children's insecurities and tendency to exaggerate mundane dilemmas.<br /><br />If you were to hyperbolize this movie the way kids do their quandaries, you will see that it is essentially about war. Freedom-fighter and freedom-hater use pubescent boys as pawns in proxy wars, only to learn a valuable lesson in unity. International leaders can learn a thing

In [11]:
train_df.tail()

,review,sentiment
24995,"My comments may be a bit of a spoiler, for what it's worth. Stop now if you care enough....<br /><br />Saving Grace should have been titled ""A Paper-Thin Excuse for Old British Women to Get High On-Screen."" This film is dumb. The incidental music is an annoyance as are the obvious, hackneyed tunes that sporadically pop up to comment on the narrative (""Spirit in the Sky,"" for example - Oh, I get it!) This is basically a Cheech and Chong movie made credible by its stodgy English setting and Brenda Blethyn's overwhelming power to inflict emotion on an audience using her voice alone. I could literally hear the folks over at High Times magazine receiving their jollies over the enormous ""buds"" that litter this picture. Worst scene? Easy. Brenda attempts to peddle her illicit wares on the street of London in a blaring white dress-suit. Not funny. Not original. Not interesting. Not a good movie. The 7.2 rating is the result of zealots over-voting. Don't waste your time...",0
24996,"The ""saucy"" misadventures of four au pairs who arrive in London on the same day in the early 1970s. There's a Swedish girl, a Danish, a German and a Chinese. The story contrives to get the clothes off all of them, involve them in some Carry On-type humour and couple them with various misfits from the British film and TV culture of the time, including Man About the House star Richard O'Sullivan, future Coronation Street rogue Johnny Briggs and horror film stalwart Ferdy Mayne (playing a sheik). There's a pretty risqué amount of female nudity on display, for those who like that kind of thing (but obviously nothing hardcore).<br /><br />Most of the film is pretty thin and inconsequential; the girls are stereotypes, and German Anita especially suffers from some kind of infantalising disorder - she's a moron obsessed with colour TV who acts like a kind of uninhibited child & dresses to deliberately show her private parts; in another more serious film, she would be a psychiatric case. The most interesting section of the film involves the Swedish girl being taken to a club in London where some dodgy types are still trying to swing, being seduced by a middle-aged rocker, losing her virginity and realising that the scene is not for her. These sequences have some energy in them and point to a more intriguing film than we've ended up with, in which promiscuity and the dregs of the music business and upper classes live soulless and seedy lives (there's a fine turn by John Standing as an impotent public school roué). The strangest of the stories has the Chinese girl (future cannibal film veteran Me Me Lay) getting off with her childish piano prodigy employer, falling mutually in love with and then leaving in the middle of the night for no good reason at all, except some orientalist notion that ""Chinese birds are inscrutable, ain't they?!"" The film is pretty demeaning to its women characters and there's a smattering of homophobia in the dialogue and one of the characterisations. The end is striking, as Mayne's sheik for no earthly reason (except they have to end the film somehow) whisks all of the girls away to his Arab kingdom for what looks to all the world like a future in the white slave trade, which they are all delighted about.<br /><br />Stuff and nonsense for the most part then, but directed with a fair amount of skill by veteran Val Guest, which puts it as a piece of film-making a notch above most of the 70s Brit sexploitation flicks.",0
24997,"Oh, those Italians! Assuming that movies about aristocrats with weird fetishes, castles drowned in gothic atmosphere, and back-stabbing relatives trying to get their hands on an inheritance are inherently interesting to all! If you've seen one film of this type, you've basically seen them all (the MST3K favorite ""Screaming Skull"" fits the mold, too)...and ""The Night Evelyn Came Out of the Grave"" is formulaic, by-the-numbers, and dull as hell. Even the luscious Erika Blanc is put to w

In [12]:
train_class_distribution = train_df["sentiment"].value_counts()

print(
    "La distribución de clases en el conjunto de entrenamiento es:",
    train_class_distribution,
)

La distribución de clases en el conjunto de entrenamiento es: sentiment
1    12500
0    12500
Name: count, dtype: int64


## Preprocesamiento de Texto

El preprocesamiento de texto es esencial para preparar los datos de texto antes de ingresarlos en un modelo de aprendizaje profundo. A continuación se describen los pasos clave que realizaremos:

1. **Limpieza del Texto de Bajo Nivel**:
   - **Eliminación de HTML**: Remover etiquetas HTML u otros elementos de markup.
   - **Eliminación de Texto entre Corchetes**: Eliminar texto entre corchetes, como [imagen], [audio], etc.
   - **Eliminación de Caracteres No AlfaNuméricos**: Remover caracteres especiales, como puntuación y otros símbolos.
   - **Eliminación de Espacios en Blanco Adicionales**: Remover espacios en blanco adicionales y espacios al principio y al final del texto.

2. **Limpieza del Texto de Alto Nivel**:
   - **Transformación a Minúsculas**: Convertir todo el texto a minúsculas para evitar duplicados.
   - **Eliminación de Stop Words**: Remover palabras comunes que no aportan información.
   - **Lematización**: Convertir las palabras a su forma base utilizando técnicas de lematización.

3. **Construcción del Vocabulario**:
   - **Creación de un Diccionario**: Asignar un índice único a cada palabra en el corpus.
   - **Filtro por Frecuencia**: Eliminar palabras demasiado raras o comunes.


### Limpieza del Texto de Bajo Nivel

En esta etapa, eliminaremos las etiquetas HTML, la puntuación, los caracteres especiales y los espacios en blanco innecesarios de las reseñas. Nos ayudaremos con expresiones regulares de la [biblioteca `re`](https://docs.python.org/3/library/re.html) para realizar estas tareas. 

In [13]:
def strip_html_tags(text):
    """Elimina etiquetas HTML"""
    pattern = r"<.*?>"
    return re.sub(pattern, "", text)

def remove_between_square_brackets(text):
    """Elimina texto entre corchetes cuadrados (ej: [Spoiler], [Citation needed])"""
    pattern = r"\[[^\]]*\]"
    return re.sub(pattern, "", text)

def remove_special_characters(text, keep_punctuation=True):
    """
    Elimina caracteres especiales
    
    Args:
        text: texto a limpiar
        keep_punctuation: si True, conserva puntuación básica (.,!?;:)
    """
    if keep_punctuation:
        # Conservamos letras, números, espacios y puntuación común
        pattern = r"[^a-zA-Z0-9\s.,!?;:\-']"
    else:
        # Solo letras, números y espacios
        pattern = r"[^a-zA-Z0-9\s]"
    
    return re.sub(pattern, "", text)

def remove_additional_whitespace(text):
    """Elimina espacios en blanco múltiples y espacios alrededor de puntuación"""
    # Primero eliminamos espacios múltiples
    text = re.sub(r"\s{2,}", " ", text)
    # Eliminamos espacios antes de puntuación
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    # Aseguramos un espacio después de puntuación (si no hay ya)
    text = re.sub(r"([.,!?;:])([^\s])", r"\1 \2", text)
    # Eliminamos espacios al inicio y final
    return text.strip()

def low_level_text_cleaning(text, keep_punctuation=True):
    """
    Limpieza completa de texto
    
    Args:
        text: texto a limpiar
        keep_punctuation: si True, conserva puntuación (recomendado para NLP)
    """
    text = strip_html_tags(text)
    text = remove_between_square_brackets(text)
    text = remove_special_characters(text, keep_punctuation)
    text = remove_additional_whitespace(text)
    return text


# Texto de prueba
testing_text = "<p>Hello    World! [Spoiler] The villain is the butler! [End of spoiler] <br> See you .</p>"
print(f"Original: {testing_text}")
print(f"Cleaned: {low_level_text_cleaning(testing_text)}")

Original: <p>Hello    World! [Spoiler] The villain is the butler! [End of spoiler] <br> See you .</p>
Cleaned: Hello World! The villain is the butler! See you.


In [14]:
# Aplicamos la limpieza a los conjuntos de entrenamiento y prueba
train_df["review_low_level_cleaned"] = train_df["review"].apply(low_level_text_cleaning)
test_df["review_low_level_cleaned"] = test_df["review"].apply(low_level_text_cleaning)

### Limpieza del Texto de Alto Nivel

En esta etapa, convertiremos el texto a minúsculas y eliminaremos las palabras vacías (stop words) del texto. Las palabras vacías son palabras comunes que no aportan información significativa al texto, como "a", "the", "is", etc. Utilizaremos la biblioteca `nltk` para descargar la lista de palabras vacías y eliminarlas del texto.

> **Nota**: Dependiendo de la tarea y el dominio, es posible que desee personalizar la lista de palabras vacías para adaptarla a sus necesidades.

> **Nota 2**: A veces descargar el paquete de stopwords puede fallar debido a `429: Too Many Requests`. Si esto ocurre, hay que descargar el paquete manualmente `python -m nltk.downloader stopwords`

In [15]:
# Descargamos stopwords de nltk
nltk.download("stopwords")
all_stopwords = set(stopwords.words("english"))

print(f"Stopwords: {all_stopwords}")

Stopwords: {"wasn't", 'we', "it'd", 'those', "weren't", 'couldn', "don't", 'you', 'was', "haven't", 'mightn', 'again', 'theirs', 'other', 'or', 'it', 'herself', 'below', 'for', "mustn't", 'they', "we're", 'she', 'aren', "we'll", 's', 'should', 'wasn', "isn't", 're', 'before', "should've", 'weren', "shouldn't", "i'd", 'doing', 'ours', 'then', 'your', 'themselves', 'do', 'further', 'down', 'can', 'once', "shan't", 'just', 'in', "hasn't", 'more', 'through', 'himself', 'but', 'her', 'here', 'd', 'o', 'ourselves', 'during', "it's", 't', 'any', 'll', 'won', 'under', 'a', 'being', 'after', "you're", 'until', 'hasn', 'them', 'didn', "they've", 'our', 'nor', 'be', 'now', "we've", 'has', 'an', 'these', 'into', 'not', 'up', "i'm", 'their', 'of', 'from', 'by', 'at', 'my', 'and', 'off', 'while', "mightn't", 'when', 'yours', 'm', "won't", 'between', "you'll", 'because', 'why', 'if', 'so', 'such', 'were', 'am', 'ma', 'very', 'hadn', "she'd", 'i', 'about', 'doesn', 'own', "needn't", 'hers', "aren't", 

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/rodrigobenitez/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [16]:
def remove_stop_words(full_text_line):
    # Eliminamos stopwords
    tokens = full_text_line.split()
    tokens = [token for token in tokens if token not in all_stopwords]
    return " ".join(tokens)

def to_lower_case(full_text_line):
    # Convertimos a minúsculas
    return full_text_line.lower()

# Texto de prueba
testing_text = "The quick brown fox don't jumps over the lazy dog"
print(f"Original:                          {testing_text}")
print(f"Lower Case and Stop Words Removed: {remove_stop_words(to_lower_case(testing_text))}")

Original:                          The quick brown fox don't jumps over the lazy dog
Lower Case and Stop Words Removed: quick brown fox jumps lazy dog


#### Lematización / Stemming

- La lematización es el proceso de convertir las palabras a su forma base o lema. Por ejemplo, las palabras "corriendo", "corre" y "corrió" se convertirían a "correr". La lematización ayuda a reducir la variabilidad de las palabras y a agrupar palabras similares juntas.

- El stemming es un proceso similar a la lematización, pero más simple. Consiste en eliminar los sufijos de las palabras para obtener su raíz. Por ejemplo, las palabras "corriendo", "corre" y "corrió" se convertirían a "corr". Aunque el stemming es más rápido que la lematización, a menudo produce resultados menos precisos.

In [17]:
# necesitamos esto para que funcione el tokenizador (nltk.word_tokenize()), de esta forma maneja mejor la puntuación
nltk.download('punkt') 
nltk.download('punkt_tab')

# necesitamos esto para que funcione el lematizador (WordNetLemmatizer)
nltk.download('wordnet')
# para que la salida de pos_tag sea compatible con WordNetLemmatizer
nltk.download('universal_tagset') 

# necesitamos esto para que funcione el etiquetador POS (nltk.pos_tag)
nltk.download('averaged_perceptron_tagger') 
nltk.download('averaged_perceptron_tagger_eng')

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/rodrigobenitez/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/rodrigobenitez/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/rodrigobenitez/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     /Users/rodrigobenitez/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/rodrigobenitez/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/rodrigobenitez/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[n

Cuando vamos a separar el texto en tokens, no es tan sencillo cómo hacer un `split(" ")`, ya que hay que tener en cuenta la puntuación, los signos de interrogación, etc. Para esto, utilizamos el tokenizador de NLTK `nltk.word_tokenize()`, que maneja estos casos de manera adecuada.

In [18]:
test_texts = [
    "Hello, world! This is a test.",
    "It's a beautiful day, isn't it?",
    "Dr. Smith went to Washington D.C. on Jan. 5th.",
    "The quick brown fox jumps over the lazy dog."
]

for testing_text in test_texts:
    tokens = nltk.word_tokenize(testing_text) # separamos en tokens
    print(f"Original:   {testing_text}")
    print(f"tokens: {tokens}\n")

Original:   Hello, world! This is a test.
tokens: ['Hello', ',', 'world', '!', 'This', 'is', 'a', 'test', '.']

Original:   It's a beautiful day, isn't it?
tokens: ['It', "'s", 'a', 'beautiful', 'day', ',', 'is', "n't", 'it', '?']

Original:   Dr. Smith went to Washington D.C. on Jan. 5th.
tokens: ['Dr.', 'Smith', 'went', 'to', 'Washington', 'D.C.', 'on', 'Jan.', '5th', '.']

Original:   The quick brown fox jumps over the lazy dog.
tokens: ['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '.']



In [19]:
# Función para convertir POS (Part of Speech) tags
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'): # adjetivo
        return wordnet.ADJ
    elif treebank_tag.startswith('V'): # verbo
        return wordnet.VERB
    elif treebank_tag.startswith('N'): # sustantivo
        return wordnet.NOUN
    elif treebank_tag.startswith('R'): # adverbio
        return wordnet.ADV
    else:
        return wordnet.NOUN  # por defecto, sustantivo

text = "The children were running faster than their friends."

print("="*80)
print("TEXTO ORIGINAL:")
print("="*80)
print(text)

# Tokenizar
tokens = nltk.word_tokenize(text.lower())  # lowercase para mejor procesamiento

# POS tagging
pos_tags = nltk.pos_tag(tokens, tagset='universal')

print("\n" + "="*80)
print("ANÁLISIS DETALLADO:")
print("="*80)
print(f"{'Original':<15} {'POS':<8} {'Lemma':<15} {'Stem':<15}")
print("-"*80)
    
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

lemmatized_tokens = []
stemmed_tokens = []
for token, pos in pos_tags:
    if token.isalpha():  # Solo palabras, ignorar puntuación
        wordnet_pos = get_wordnet_pos(pos)
        lemma = lemmatizer.lemmatize(token, wordnet_pos)
        stem = stemmer.stem(token)
        
        # Mostrar solo si hay cambio
        if token != lemma or token != stem:
            print(f"{token:<15} {pos:<8} {lemma:<15} {stem:<15}")
        
        lemmatized_tokens.append(lemma)
        stemmed_tokens.append(stem)
    else:
        lemmatized_tokens.append(token)
        stemmed_tokens.append(token)

print("\n" + "="*80)
print("TEXTO LEMATIZADO:")
print("="*80)
lemmatized_text = " ".join(lemmatized_tokens).replace(".", ".\n")
print(lemmatized_text)

print("\n" + "="*80)
print("TEXTO STEMMED:")
print("="*80)
stemmed_text = " ".join(stemmed_tokens).replace(" .", ".\n")
print(stemmed_text)


TEXTO ORIGINAL:
The children were running faster than their friends.

ANÁLISIS DETALLADO:
Original        POS      Lemma           Stem           
--------------------------------------------------------------------------------
children        NOUN     child           children       
were            VERB     be              were           
running         VERB     run             run            
friends         NOUN     friend          friend         

TEXTO LEMATIZADO:
the child be run faster than their friend .


TEXTO STEMMED:
the children were run faster than their friend.



In [20]:
def lemmatize(text):
    # Tokenizar el texto
    tokens = nltk.word_tokenize(text)
    # Obtener etiquetas POS
    pos_tags = nltk.pos_tag(tokens, tagset='universal')
    # Lematizar cada token con su etiqueta POS correspondiente
    lemmatized_tokens = [lemmatizer.lemmatize(token, get_wordnet_pos(pos_tag)) for token, pos_tag in pos_tags]
    return " ".join(lemmatized_tokens)

def stem_words(text):
    # Aplicamos stemming a cada palabra
    tokens = nltk.word_tokenize(text)
    stemmed_tokens = [stemmer.stem(token) for token in tokens]
    return " ".join(stemmed_tokens)

def high_level_text_cleaning(text, remove_stop_words=False):
    text = to_lower_case(text)
    if remove_stop_words:
        text = remove_stop_words(text)
    text = lemmatize(text)
    return text

def high_level_text_cleaning_v2(text, remove_stop_words=False):
    text = to_lower_case(text)
    if remove_stop_words:
        text = remove_stop_words(text)
    text = stem_words(text)
    return text

print(f"Original: {text}")
print(f"Lemma: {high_level_text_cleaning(text)}")
print(f"Steam: {high_level_text_cleaning_v2(text)}")

Original: The children were running faster than their friends.
Lemma: the child be run faster than their friend .
Steam: the children were run faster than their friend .


In [21]:
testing_text = (
    "Why waste time saying a lot of words when a few words do the trick?"
)
print(f"Lemma: {high_level_text_cleaning(testing_text)}")
print(f"Steam: {high_level_text_cleaning_v2(testing_text)}")

Lemma: why waste time say a lot of word when a few word do the trick ?
Steam: whi wast time say a lot of word when a few word do the trick ?


<img src="https://media1.tenor.com/m/IsYdPRq7bjcAAAAC/why-waste-time-when-few-word-do-trick.gif"/>

[Kevin's Small Talk - The Office US](https://www.youtube.com/watch?v=_K-L9uhsBLM)

In [22]:
# Aplicamos la limpieza de alto nivel a los conjuntos de entrenamiento y prueba
train_df["review_clean"] = train_df["review_low_level_cleaned"].apply(high_level_text_cleaning_v2) # podemos usar lemmatization o stemming
test_df["review_clean"] = test_df["review_low_level_cleaned"].apply(high_level_text_cleaning_v2)

### Construcción del Vocabulario

Finalmente, construiremos un vocabulario a partir de las reseñas limpias. Un vocabulario es un conjunto de todas las palabras únicas en el corpus. Cada palabra en el vocabulario se asigna a un índice único, que se utilizará para convertir el texto en una secuencia de índices numéricos.

In [23]:
OOV_TOKEN = "<OOV>"
PAD_TOKEN = "<PAD>"
MAX_VOCAB_SIZE = 20_000
SEQUENCE_LENGTH = 200
EMBEDDING_DIM = 100

In [24]:
def make_vocab(all_texts, max_vocab_size, min_freq=5):
    # Contamos la frecuencia de cada palabra
    counts = Counter(chain(*(all_texts.str.split())))
    counts = {word: freq for word, freq in counts.items() if freq >= min_freq}

    # Ordenamos las palabras por frecuencia y nos quedamos con las max_vocab_size palabras más frecuentes
    vocab = sorted(counts, key=counts.get, reverse=True)[:max_vocab_size]
    vocab.append(OOV_TOKEN)  # Añadimos el token OOV al final

    # Mapa de palabras a índices
    vocab_to_int = {word: ii for ii, word in enumerate(vocab, 1)}
    vocab_to_int[PAD_TOKEN] = 0  # Añadimos el token PAD al principio

    return vocab_to_int

# Texto de prueba
testing_text1 = "The quick brown fox jumps over the lazy dog"
testing_text2 = "Then the quick brown fox jumps over the lazy dog"
print(f"Vocab: {make_vocab(pd.Series([testing_text1, testing_text2]), 100, 1)}")

Vocab: {'the': 1, 'quick': 2, 'brown': 3, 'fox': 4, 'jumps': 5, 'over': 6, 'lazy': 7, 'dog': 8, 'The': 9, 'Then': 10, '<OOV>': 11, '<PAD>': 0}


In [25]:
word_to_index = make_vocab(train_df["review_clean"], MAX_VOCAB_SIZE)
VOCAB_SIZE = len(word_to_index)
print(f"Tamaño del vocabulario: {VOCAB_SIZE}")

Tamaño del vocabulario: 20002


Ahora vamos a implementar una funcion que transforma un string con la review en una lista de enteros con la posición de las palabras en el vocabulario.

In [26]:
def get_review_features(review_text, word_to_idx):
    """
    Convierte un texto en una lista de índices basados en el vocabulario.
    """
    # Tokenizar el texto y convertir cada palabra a su índice correspondiente
    return [
        word_to_idx.get(word, word_to_idx[OOV_TOKEN]) for word in review_text.split()
    ]
    
def truncate_sequence(sequence, max_length, keep='last'):
    """
    Trunca la secuencia manteniendo las primeras o últimas palabras
    
    Args:
        sequence: lista de tokens/índices
        max_length: longitud máxima
        keep: 'first' o 'last'
    """
    if len(sequence) <= max_length:
        return sequence
    
    if keep == 'last':
        return sequence[-max_length:]  # Últimas palabras
    else:
        return sequence[:max_length]   # Primeras palabras

def left_pad_features(review_ints, seq_length, pad_value=0, truncate_keep='last'):
    """
    Aplica padding a la izquierda a una secuencia de índices para que todas las secuencias tengan la misma longitud.
    """
    # Truncar si es más largo que seq_length
    review_ints = truncate_sequence(review_ints, seq_length, keep=truncate_keep)
    
    # Padding a la izquierda
    if len(review_ints) < seq_length:
        padding = [pad_value] * (seq_length - len(review_ints))
        return padding + review_ints
    else:
        return review_ints

def get_review_representation(review_text, word_to_idx, max_sequence_length):
    """
    Convierte el texto de entrada en una representación de secuencia con padding a la izquierda.
    """
    review_ints = get_review_features(review_text, word_to_idx)
    return left_pad_features(review_ints, max_sequence_length)

# Texto de prueba
testing_text = "The quick brown fox jumps <br> over the lazy dog ! ThisWordIsOOV."
print(f"Original: {testing_text}")
testing_text = low_level_text_cleaning(testing_text)
print(f"Limpieza bajo nivel: {testing_text}")
testing_text = high_level_text_cleaning(testing_text)
print(f"Limpieza alto nivel: {testing_text}")
testing_text = get_review_representation(testing_text, word_to_index, 15)
print(f"Features: {testing_text}")

Original: The quick brown fox jumps <br> over the lazy dog ! ThisWordIsOOV.
Limpieza bajo nivel: The quick brown fox jumps over the lazy dog! ThisWordIsOOV.
Limpieza alto nivel: the quick brown fox jump over the lazy dog ! thiswordisoov .
Features: [0, 0, 0, 1, 1627, 1744, 1507, 959, 147, 1, 20001, 753, 32, 20001, 2]


## Dataset y DataLoader

Nuestro dataset `IMDBDataset` tomará el DataFrame de Pandas con las **reseñas preprocesadas** y el vocabulario, y devolverá una secuencia de índices numéricos para cada reseña. Luego, utilizaremos un DataLoader para cargar los datos en lotes y alimentarlos a nuestro modelo.

In [27]:
class IMDBDataset(Dataset):
    def __init__(self, reviews, labels, vocab, max_sequence_length):
        self.reviews = reviews
        self.labels = labels
        self.vocab = vocab
        self.max_sequence_length = max_sequence_length

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, idx):
        # Obtener texto y etiqueta
        text = self.reviews[idx]
        label = self.labels[idx]

        # Convertir texto a representación con padding
        indices = get_review_representation(text, self.vocab, self.max_sequence_length)

        return torch.tensor(indices, dtype=torch.int32), torch.tensor(
            [label], dtype=torch.float32
        )

In [28]:
# Crear el dataset
train_dataset = IMDBDataset(
    train_df["review_clean"],
    train_df["sentiment"],
    word_to_index,
    max_sequence_length=SEQUENCE_LENGTH,
)
test_dataset = IMDBDataset(
    test_df["review_clean"],
    test_df["sentiment"],
    word_to_index,
    max_sequence_length=SEQUENCE_LENGTH,
)

# Separar el conjunto de entrenamiento en subconjuntos de entrenamiento y validación
train_len = len(train_dataset)
val_len = int(0.2 * train_len)
train_dataset, val_dataset = random_split(train_dataset, [train_len - val_len, val_len])

In [29]:
train_dataset[0]

(tensor([ 3902,   149,    40,  2792,     2,     1,   406,     6,     1,    17,
            14,   190,   396,   697,  2030,    36,   705,     4,  6017,  2338,
            28,   207,    55,    18,    87,     3,     4,  2792,   342,   769,
            91,    44,   175,  3564,   512,   770,    10,     1,  1010,   232,
           119,     1,   406,     6,     1,    15,    48,  2486,     2,  2792,
            14,  4747,   149,   427,   273,    10,     1,  1750,     4,    20,
            31,    57,    90,    34,    22,     7,  1246,   139,     3,     4,
            56,    68, 12874,   512,    37,  1443,  6500,   123,   265,   143,
            88,    35,    27,     1,   219,     7,     2,  5867,    14,  3564,
           512,   342,   685, 20001,     3,   215,     3,    13,   226,    27,
            97,  1033,   262,    29,     8,   111,     5,  1598,    10,  3719,
             2,   172,   487,   767,    10,    53,   746,     9,   405,    83,
          1483,     1,  1220,    10,    73,     9,  

In [30]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Parte 1: Modelos sin RNNs

## Modelo base

Para capturar la semántica de las reseñas necesitamos tomar nuestro texto (ya convertido a índices) y convertirlo en un vector de características. Para esto utilizamos una capa de embedding que mapea cada índice a un vector de características. 


### nn.Embedding

La [capa de embedding](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html) en PyTorch es una capa lineal que mapea un índice a un vector de características. Por ejemplo, si nuestro vocabulario tiene 10,000 palabras y estamos utilizando un embedding de tamaño 100, la capa de embedding tendrá una matriz de pesos de tamaño 10,000 x 100. Dado un índice de palabra, la capa de embedding devuelve la fila correspondiente de la matriz de pesos, que es el vector de características de la palabra.

In [31]:
embedding_layer = nn.Embedding(
    VOCAB_SIZE, EMBEDDING_DIM, padding_idx=word_to_index[PAD_TOKEN]
)
word_indices = torch.tensor([0, 1, 2, 3, 4, 5])
embedding_layer(word_indices)

tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.

Al igual que con otras capas en PyTorch, la capa de embedding se inicializa con pesos aleatorios y se ajusta durante el entrenamiento.


$$
\text{Parámetros} = \text{Tamaño del Vocabulario} \times \text{Tamaño del Embedding} + \text{Tamaño del Embedding}
$$

### Arquitectura

Para clasificar las reseñas de IMDB, utilizaremos una arquitectura de modelo simple con las siguientes capas:

1. **Capa de Embedding**: Mapea cada índice de palabra a un vector de características.
2. **Capa de Promedio**: Calcula el promedio de los vectores de características de todas las palabras en una reseña.
3. **Capa Lineal Oculta**: Transforma el vector de características promedio en un vector de características de tamaño oculto.
4. **Capa de Salida**: Produce la salida final, que es la probabilidad de que la reseña sea positiva o negativa.

In [32]:
class SentimentModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(SentimentModel, self).__init__()
        self.embed = nn.Embedding(vocab_size, embedding_dim, padding_idx=word_to_index[PAD_TOKEN])
        self.fc = nn.Linear(embedding_dim, hidden_dim)
        self.relu = nn.ReLU(inplace=True)
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x: [BATCH_SIZE, SEQUENCE_LENGTH]
        embed = self.embed(x)
        # x: [BATCH_SIZE, SEQUENCE_LENGTH, EMBEDDING_DIM]
        x = torch.mean(embed, dim=1)
        # x: [BATCH_SIZE, SEQUENCE_LENGTH]
        x = self.relu(self.fc(x))
        return F.sigmoid(self.out(x))

summary(
    SentimentModel(VOCAB_SIZE, EMBEDDING_DIM, 512),
    input_size=(BATCH_SIZE, SEQUENCE_LENGTH),
    dtypes=[torch.int32],
)

Layer (type:depth-idx)                   Output Shape              Param #
SentimentModel                           [512, 1]                  --
├─Embedding: 1-1                         [512, 200, 100]           2,000,200
├─Linear: 1-2                            [512, 512]                51,712
├─ReLU: 1-3                              [512, 512]                --
├─Linear: 1-4                            [512, 1]                  513
Total params: 2,052,425
Trainable params: 2,052,425
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 1.05
Input size (MB): 0.41
Forward/backward pass size (MB): 84.02
Params size (MB): 8.21
Estimated Total Size (MB): 92.64

### Entrenamiento y Evaluación

In [33]:
CRITERION = nn.BCELoss().to(DEVICE)

In [34]:
base_model = SentimentModel(
    vocab_size=VOCAB_SIZE, embedding_dim=EMBEDDING_DIM, hidden_dim=512
).to(DEVICE)
base_optimizer = optim.Adam(base_model.parameters(), lr=0.001)

In [35]:
_, _ = train(
    base_model,
    optimizer=base_optimizer,
    criterion=CRITERION,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    do_early_stopping=True,
    patience=3,
    epochs=20,
)

Epoch: 001 | Train Loss: 0.66358 | Val Loss: 0.62000
Epoch: 002 | Train Loss: 0.56337 | Val Loss: 0.50912
Epoch: 003 | Train Loss: 0.45856 | Val Loss: 0.43977
Epoch: 004 | Train Loss: 0.38859 | Val Loss: 0.39622
Epoch: 005 | Train Loss: 0.33517 | Val Loss: 0.37058
Epoch: 006 | Train Loss: 0.29840 | Val Loss: 0.36936
Epoch: 007 | Train Loss: 0.27069 | Val Loss: 0.35456
Epoch: 008 | Train Loss: 0.24405 | Val Loss: 0.34066
Epoch: 009 | Train Loss: 0.21990 | Val Loss: 0.33771
Epoch: 010 | Train Loss: 0.19737 | Val Loss: 0.33979
Epoch: 011 | Train Loss: 0.18029 | Val Loss: 0.34840
Epoch: 012 | Train Loss: 0.16714 | Val Loss: 0.34224
Detener entrenamiento en la época 11, la mejor pérdida fue 0.33771


In [36]:
def model_accuracy(model, data_loader):
    model.eval()
    with torch.no_grad():
        y_true = []
        y_pred = []
        for x, y in data_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = torch.where(model(x) > 0.5, 1, 0)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(out.cpu().numpy())
        print(f"Accuracy: {np.mean(np.array(y_true) == np.array(y_pred)) * 100:.2f}%")

In [37]:
model_accuracy(base_model, test_loader)

Accuracy: 85.36%


## Ejercicios

- Explorar otras técnicas de preprocesamiento de texto, como stemming, eliminación de números, etc.
- Explorar con los hiperparámetros del modelo, como el tamaño del embedding, el tamaño de la capa oculta, etc.

# Parte 2: Modelos con RNNs

Las [redes neuronales recurrentes (RNNs)](https://d2l.ai/chapter_recurrent-neural-networks/rnn.html) son una clase de redes neuronales diseñadas para manejar datos secuenciales. A diferencia de las redes neuronales convolucionales (CNNs), que son eficaces para procesar datos espaciales, como imágenes, las RNNs son ideales para modelar datos secuenciales, como texto, audio y series temporales.

En este caso tendremos una arquitectura many-to-one, donde la entrada es una secuencia de palabras y la salida es una sola etiqueta de clase (positiva o negativa).

## RNNs

### nn.RNN

La capa [RNN](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html) en PyTorch es una capa recurrente que procesa una secuencia de entrada paso a paso, manteniendo un estado oculto que captura la información de pasos anteriores. Dado un tensor de entrada de tamaño `(batch, secuencia, características)`, la capa RNN procesa la secuencia paso a paso y devuelve el estado oculto final para cada secuencia en el lote.

<!-- ![rnn](https://d2l.ai/_images/rnn.svg) -->
<img src="https://d2l.ai/_images/rnn.svg" width="500" style="background:white; display: block; margin-left: auto; margin-right: auto;"/>

In [38]:
rnn = nn.RNN(input_size=EMBEDDING_DIM, hidden_size=32, num_layers=1, batch_first=True)
tensor = torch.randn(BATCH_SIZE, SEQUENCE_LENGTH, EMBEDDING_DIM)

output, hidden = rnn(tensor)
print(f"Output shape: {output.shape}")
print(f"Hidden shape: {hidden.shape}")

Output shape: torch.Size([512, 200, 32])
Hidden shape: torch.Size([1, 512, 32])


### Arquitectura

In [40]:
from torch import embedding

class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, n_layers=2, dropout=0.5):
        super(SentimentRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=word_to_index[PAD_TOKEN])
        self.drop = nn.Dropout(dropout)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers=n_layers, dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)  # Assuming binary classification

    def forward(self, x):
        # x: [BATCH_SIZE, SEQUENCE_LENGTH]
        embeds = self.embedding(x)
        embeds = self.drop(embeds)
        rnn_out, hidden = self.rnn(embeds)
        out = F.relu(hidden[-1])
        out = self.drop(out)
        out = self.fc(out)
        return torch.sigmoid(out)

summary(
    SentimentRNN(VOCAB_SIZE, EMBEDDING_DIM, 128),
    input_size=(BATCH_SIZE, SEQUENCE_LENGTH),
    dtypes=[torch.int32],
)

Layer (type:depth-idx)                   Output Shape              Param #
SentimentRNN                             [512, 1]                  --
├─Embedding: 1-1                         [512, 200, 100]           2,000,200
├─Dropout: 1-2                           [512, 200, 100]           --
├─RNN: 1-3                               [512, 200, 128]           62,464
├─Dropout: 1-4                           [512, 128]                --
├─Linear: 1-5                            [512, 1]                  129
Total params: 2,062,793
Trainable params: 2,062,793
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 7.42
Input size (MB): 0.41
Forward/backward pass size (MB): 186.78
Params size (MB): 8.25
Estimated Total Size (MB): 195.44

### Entrenamiento y Evaluación

In [41]:
rnn_model = SentimentRNN(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=128,
    n_layers=2,
    dropout=0.5,
).to(DEVICE)
rnn_optimizer = optim.Adam(rnn_model.parameters(), lr=0.001)

In [ ]:
_, _ = train(
    rnn_model,
    optimizer=rnn_optimizer,
    criterion=CRITERION,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    do_early_stopping=False,
    epochs=20,
)

In [ ]:
model_accuracy(rnn_model, test_loader)

## LSTMs

Las [redes LSTM (Long Short-Term Memory)](https://d2l.ai/chapter_recurrent-modern/lstm.html) son una variante de las RNNs que están diseñadas para manejar el problema del desvanecimiento del gradiente. Las LSTMs utilizan una estructura de celda más compleja que permite que el gradiente fluya sin desvanecerse o explotar, lo que las hace más efectivas para modelar secuencias a largo plazo.

<img src="https://d2l.ai/_images/lstm-3.svg" width="500" style="background:white; display: block; margin-left: auto; margin-right: auto;"/>

In [43]:
lstm = nn.LSTM(input_size=EMBEDDING_DIM, hidden_size=32, num_layers=1, batch_first=True)
tensor = torch.randn(BATCH_SIZE, SEQUENCE_LENGTH, EMBEDDING_DIM)

output, (hidden, cell) = lstm(tensor)
print(f"Output shape: {output.shape} (batch_size, seq_length, hidden_size)")
print(f"Hidden shape: {hidden.shape} (num_layers, batch_size, hidden_size)")
print(f"Cell shape: {cell.shape} (num_layers, batch_size, hidden_size)")

Output shape: torch.Size([512, 200, 32]) (batch_size, seq_length, hidden_size)
Hidden shape: torch.Size([1, 512, 32]) (num_layers, batch_size, hidden_size)
Cell shape: torch.Size([1, 512, 32]) (num_layers, batch_size, hidden_size)


In [44]:
class SentimentLSTM(nn.Module):
    def __init__(
        self, vocab_size, embedding_dim, hidden_dim, n_layers=2, dropout=0.5
    ):
        super(SentimentLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=word_to_index[PAD_TOKEN])
        self.drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=n_layers, dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)  # Assuming binary classification

    def forward(self, x):
        # x: [BATCH_SIZE, SEQUENCE_LENGTH]
        embeds = self.embedding(x)
        embeds = self.drop(embeds)
        rnn_out, (hidden, _) = self.lstm(embeds)
        out = F.relu(hidden[-1])
        out = self.drop(out)
        out = self.fc(out)
        return torch.sigmoid(out)

summary(
    SentimentLSTM(VOCAB_SIZE, EMBEDDING_DIM, 128),
    input_size=(BATCH_SIZE, SEQUENCE_LENGTH),
    dtypes=[torch.int32],
)

Layer (type:depth-idx)                   Output Shape              Param #
SentimentLSTM                            [512, 1]                  --
├─Embedding: 1-1                         [512, 200, 100]           2,000,200
├─Dropout: 1-2                           [512, 200, 100]           --
├─LSTM: 1-3                              [512, 200, 128]           249,856
├─Dropout: 1-4                           [512, 128]                --
├─Linear: 1-5                            [512, 1]                  129
Total params: 2,250,185
Trainable params: 2,250,185
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 26.61
Input size (MB): 0.41
Forward/backward pass size (MB): 186.78
Params size (MB): 9.00
Estimated Total Size (MB): 196.19

In [ ]:
lstm_model = SentimentLSTM(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=128,
    n_layers=4,
    dropout=0.5,
).to(DEVICE)

optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)

_, _ = train(
    lstm_model,
    optimizer=optimizer,
    criterion=CRITERION,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    do_early_stopping=False,
    epochs=20,
)

In [ ]:
model_accuracy(lstm_model, test_loader)

## Ejericios

- Explorar otras arquitecturas de RNNs, como [GRUs](https://d2l.ai/chapter_recurrent-modern/gru.html).
- Experimentar con diferentes hiperparámetros, como el tamaño de la capa oculta, la tasa de aprendizaje, etc.